# News Timestamp and Temporal-Alignment Audit

This notebook performs **no API requests and no model training**. It verifies the timing convention requested by Dr. Zhang before the Phase 0-5 rerun.

Protocol:
- Alpha Vantage `time_published` is treated as a UTC provider timestamp.
- `published_at_utc` must equal the strict parse of `time_published` exactly.
- `pub_date` is only the UTC-midnight daily bucket and must equal `floor(published_at_utc, 1 day)`.
- News assigned to query day *t* must satisfy `t 00:00:00 UTC <= published_at_utc < (t+1) 00:00:00 UTC`.
- The forecast is formed immediately after UTC day *t* is complete and predicts `log(Close[t+1]/Close[t])`.


In [1]:

import json
from pathlib import Path
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

PROJECT_DIR = Path('/content/drive/MyDrive/Crypto_Research')
NEWS_DIR = PROJECT_DIR / 'data' / 'alphavantage_news'
MARKET_DIR = PROJECT_DIR / 'data' / 'market'
OUTPUT_DIR = PROJECT_DIR / 'revised_outputs_v4' / 'final_revision_audit'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


Mounted at /content/drive


In [2]:

# FINAL REVISION: strict provider-timestamp audit. This cell reads existing CSVs only.
def audit_news_file(path: Path):
    frame = pd.read_csv(path)
    required = ['time_published', 'published_at_utc', 'pub_date']
    missing = [c for c in required if c not in frame.columns]
    if missing:
        raise ValueError(f'{path.name}: missing timestamp columns {missing}')
    raw = frame['time_published'].astype('string')
    parsed = pd.to_datetime(raw, format='%Y%m%dT%H%M%S', errors='coerce').dt.tz_localize('UTC')
    stored = pd.to_datetime(frame['published_at_utc'], errors='coerce', utc=True)
    day = pd.to_datetime(frame['pub_date'], errors='coerce', utc=True)
    invalid = parsed.isna()
    exact_mismatch = parsed.notna() & stored.notna() & (parsed != stored)
    day_mismatch = day.notna() & stored.notna() & (day != stored.dt.floor('D'))
    return {
        'file': path.name,
        'rows': len(frame),
        'invalid_raw_timestamp': int(invalid.sum()),
        'exact_utc_mismatch': int(exact_mismatch.sum()),
        'pub_date_floor_mismatch': int(day_mismatch.sum()),
        'min_utc': stored.min(), 'max_utc': stored.max(),
        'hour00_rows': int((stored.dt.hour == 0).sum()),
        'hour23_rows': int((stored.dt.hour == 23).sum()),
        'status': 'PASS' if int(invalid.sum()+exact_mismatch.sum()+day_mismatch.sum()) == 0 else 'FAIL',
    }

news_files = [
    NEWS_DIR / 'alphavantage_general_articles_raw.csv',
    NEWS_DIR / 'alphavantage_BTC_crypto_articles_raw.csv',
    NEWS_DIR / 'alphavantage_ETH_crypto_articles_raw.csv',
]
audit = pd.DataFrame([audit_news_file(p) for p in news_files])
display(audit)
audit.to_csv(OUTPUT_DIR / 'timestamp_convention_audit.csv', index=False)
if (audit['status'] != 'PASS').any():
    raise ValueError('Timestamp audit failed; do not rerun the modeling phases until resolved.')


,file,rows,invalid_raw_timestamp,exact_utc_mismatch,pub_date_floor_mismatch,min_utc,max_utc,hour00_rows,hour23_rows,status
0,alphavantage_general_articles_raw.csv,1398093,0,0,0,2022-04-01 03:19:00+00:00,2026-08-31 23:58:27+00:00,47515,36857,PASS
1,alphavantage_BTC_crypto_articles_raw.csv,61937,0,0,0,2022-04-01 00:08:13+00:00,2026-08-31 23:06:46+00:00,758,1008,PASS
2,alphavantage_ETH_crypto_articles_raw.csv,41682,0,0,0,2022-04-01 01:15:18+00:00,2026-08-31 22:30:00+00:00,497,678,PASS


In [3]:

# FINAL REVISION: verify market target construction and the allowed-news cutoff mechanically.
market_rows = []
for asset in ['BTC', 'ETH']:
    m = pd.read_csv(MARKET_DIR / f'{asset}_market.csv')
    m['Date'] = pd.to_datetime(m['Date'], errors='coerce', utc=True)
    m['Target_Date'] = pd.to_datetime(m['Target_Date'], errors='coerce', utc=True)
    calc = np.log(pd.to_numeric(m['Close'], errors='coerce').shift(-1) / pd.to_numeric(m['Close'], errors='coerce'))
    stored = pd.to_numeric(m['Target_Log_Return'], errors='coerce')
    max_target_error = float(np.nanmax(np.abs(calc - stored)))
    bad_target_dates = int(((m['Target_Date'] <= m['Date']) & m['Target_Date'].notna()).sum())
    market_rows.append({'asset': asset, 'max_abs_target_reconstruction_error': max_target_error, 'nonforward_target_dates': bad_target_dates})
market_audit = pd.DataFrame(market_rows)
display(market_audit)
market_audit.to_csv(OUTPUT_DIR / 'market_target_alignment_audit.csv', index=False)

# For every article, its forecast-origin cutoff is the next UTC midnight after its assigned day.
news_boundary_rows = []
for path in news_files:
    f = pd.read_csv(path)
    pub = pd.to_datetime(f['published_at_utc'], errors='coerce', utc=True)
    day = pub.dt.floor('D')
    origin = day + pd.Timedelta(days=1)
    violations = int(((pub < day) | (pub >= origin)).sum())
    news_boundary_rows.append({'file': path.name, 'rows': len(f), 'outside_assigned_utc_day': violations})
boundary_audit = pd.DataFrame(news_boundary_rows)
display(boundary_audit)
boundary_audit.to_csv(OUTPUT_DIR / 'news_forecast_origin_boundary_audit.csv', index=False)


,asset,max_abs_target_reconstruction_error,nonforward_target_dates
0,BTC,9.996344e-17,0
1,ETH,3.087808e-16,0


,file,rows,outside_assigned_utc_day
0,alphavantage_general_articles_raw.csv,1398093,0
1,alphavantage_BTC_crypto_articles_raw.csv,61937,0
2,alphavantage_ETH_crypto_articles_raw.csv,41682,0


In [4]:

protocol = {
    'timezone': 'UTC',
    'news_window_for_query_day_t': '[t 00:00:00 UTC, t+1 00:00:00 UTC)',
    'forecast_origin': 't+1 00:00:00 UTC after day t is complete',
    'target': 'log(Close[t+1]/Close[t])',
    'target_interval': 'UTC day-t close to UTC day-(t+1) close',
    'split_key': 'Target_Date',
    'raw_timestamp_field': 'time_published',
    'exact_parsed_field': 'published_at_utc',
    'daily_bucket_field': 'pub_date',
}
with (OUTPUT_DIR / 'temporal_alignment_protocol.json').open('w', encoding='utf-8') as handle:
    json.dump(protocol, handle, indent=2)
print(json.dumps(protocol, indent=2))


{
  "timezone": "UTC",
  "news_window_for_query_day_t": "[t 00:00:00 UTC, t+1 00:00:00 UTC)",
  "forecast_origin": "t+1 00:00:00 UTC after day t is complete",
  "target": "log(Close[t+1]/Close[t])",
  "target_interval": "UTC day-t close to UTC day-(t+1) close",
  "split_key": "Target_Date",
  "raw_timestamp_field": "time_published",
  "exact_parsed_field": "published_at_utc",
  "daily_bucket_field": "pub_date"
}
